# 27. The stack, with the four extra XGBoost seeds

**One variable against ledger row 39** (`stack_logit_25_oof`, CV 0.967873): the same
logistic combiner, the same `C`, the same fold-wise protocol, the same folds. The
member set goes from twenty-five to twenty-nine, adding the four XGBoost seeds from
ledger rows 40 to 43. Row 39 is refit in this run rather than quoted.

## The prediction, recorded before the run

Row 32 ran this exact experiment for CatBoost, adding four seeds to a stack that
already had one, and measured **+0.000014**: certain and negligible. It found that the
combiner **split** the single CatBoost weight across the five rather than adding to it,
which is what a seed spread of 1.32e-05 should produce.

Two things point the other way here, and one points the same way.

- **XGBoost's seed spread is 4.87e-05**, about 3.7x CatBoost's. Seed averaging is worth
  more when the seeds disagree more, so a larger gain than +0.000014 is expected.
- **`xgb_te` carries +0.2751**, far more weight than `cat42` carried at the equivalent
  point (+0.3512 before its seeds arrived, falling to +0.0753 after). There is more
  weight available to split.
- Against both: **row 39 showed the stack is in a substitution regime**, where a new
  member takes weight from existing ones rather than adding. Four near-copies of a
  member already present is the case where substitution should be most complete.

**The prediction is a gain between +0.000014 and +0.000050, and the mechanism is
splitting rather than adding.** Written before running so it can be scored.

The same pre-registered floor applies as in `26`: at least 4 of 5 folds positive and at
least +0.00005 on the mean, or this is logged as real-but-negligible in the way row 32
was rather than as an improvement.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def find_repo():
    for b in [Path.cwd(), *Path.cwd().parents]:
        if (b / "data" / "raw" / "train.csv").exists():
            return b
    raise FileNotFoundError("data/raw/train.csv not found")


REPO = find_repo()
O, S = REPO / "artifacts" / "oof", REPO / "submissions"

train = pd.read_csv(REPO / "data" / "raw" / "train.csv")
test = pd.read_csv(REPO / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy()

# The same split every vector on disk was produced under. Rebuilt rather than loaded,
# and then checked, because a silently different fold vector is the one error here
# that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")

train 691,369  test 296,302  folds [138274 138274 138274 138274 138273]


In [2]:
# Row 32's twenty-three in row 32's order, then the target-encoded neural model.
MEM = [
    ("te42", O / "te_bag42_oof.npy", O / "te_bag42_test.npy"),
    ("te2024", O / "te_seed2024_oof.npy", O / "te_seed2024_test.npy"),
    ("te7", O / "te_seed7_oof.npy", O / "te_seed7_test.npy"),
    ("te2025", O / "te_seed2025_oof.npy", O / "te_seed2025_test.npy"),
    ("te13", O / "te_seed13_oof.npy", O / "te_seed13_test.npy"),
    ("anchor", O / "lgbm_default_anchor_seed42.npy",
     S / "lgbm_default_anchor_seed42.csv"),
    ("trees300", O / "lgbm_trees300_seed42.npy", S / "lgbm_trees300_seed42.csv"),
    ("trees1000", O / "lgbm_trees1000_seed42.npy", S / "lgbm_trees1000_seed42.csv"),
    ("trees2000", O / "lgbm_trees2000_seed42.npy", S / "lgbm_trees2000_seed42.csv"),
    ("lr010", O / "lgbm_lr01_n1000_seed42.npy", S / "lgbm_lr01_n1000_seed42.csv"),
    ("lr005", O / "lgbm_lr005_n2000_seed42.npy", S / "lgbm_lr005_n2000_seed42.csv"),
    ("lr003", O / "lgbm_lr003_n3333_seed42.npy", S / "lgbm_lr003_n3333_seed42.csv"),
    ("bag42", O / "lgbm_bag08_lr005_n2000_seed42.npy",
     S / "lgbm_bag08_lr005_n2000_seed42.csv"),
    ("bag2024", O / "lgbm_bag08_lr005_n2000_seed2024.npy",
     S / "lgbm_bag08_lr005_n2000_seed2024.csv"),
    ("bag7", O / "lgbm_bag08_lr005_n2000_seed7.npy",
     S / "lgbm_bag08_lr005_n2000_seed7.csv"),
    ("bag2025", O / "lgbm_bag08_lr005_n2000_seed2025.npy",
     S / "lgbm_bag08_lr005_n2000_seed2025.csv"),
    ("bag13", O / "lgbm_bag08_lr005_n2000_seed13.npy",
     S / "lgbm_bag08_lr005_n2000_seed13.csv"),
    ("neural", O / "neural_oof.npy", O / "neural_test.npy"),
    ("cat42", O / "catboost_te_oof.npy", O / "catboost_te_test.npy"),
    ("cat2024", O / "catboost_te_seed2024_oof.npy",
     O / "catboost_te_seed2024_test.npy"),
    ("cat7", O / "catboost_te_seed7_oof.npy", O / "catboost_te_seed7_test.npy"),
    ("cat2025", O / "catboost_te_seed2025_oof.npy",
     O / "catboost_te_seed2025_test.npy"),
    ("cat13", O / "catboost_te_seed13_oof.npy", O / "catboost_te_seed13_test.npy"),
    ("neural_te", O / "neural_te_oof.npy", O / "neural_te_test.npy"),
    ("xgb_te", O / "xgb_te_oof.npy", O / "xgb_te_test.npy"),
    ("xgb2024", O / "xgb_te_seed2024_oof.npy", O / "xgb_te_seed2024_test.npy"),
    ("xgb7", O / "xgb_te_seed7_oof.npy", O / "xgb_te_seed7_test.npy"),
    ("xgb2025", O / "xgb_te_seed2025_oof.npy", O / "xgb_te_seed2025_test.npy"),
    ("xgb13", O / "xgb_te_seed13_oof.npy", O / "xgb_te_seed13_test.npy"),
]
NEW = ["xgb2024", "xgb7", "xgb2025", "xgb13"]
CATS = ["cat42", "cat2024", "cat7", "cat2025", "cat13"]


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


def load_test(path):
    if path.suffix == ".npy":
        return np.load(path)
    df = pd.read_csv(path)
    # A csv written in a different row order would blend perfectly cleanly and be
    # undetectable in the score. Checked rather than assumed.
    assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {path.name}"
    return df["addicted_label"].to_numpy()


names = [m[0] for m in MEM]
Poof = {n: np.load(p) for n, p, _ in MEM}
Ptest = {n: load_test(t) for n, _, t in MEM}

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])
KEEP25 = [i for i, n in enumerate(names) if n not in NEW]
print(f"{len(names)} members, oof {Loof.shape}, test {Ltest.shape}")
print("member CV:")
for n in names:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    print(f"  {n:10} {cv:.6f}" + ("   <- new" if n in NEW else ""))

29 members, oof (691369, 29), test (296302, 29)
member CV:


  te42       0.966782


  te2024     0.966771


  te7        0.966729


  te2025     0.966743


...

  xgb_te     0.967099


  xgb2024    0.967148   <- new


  xgb7       0.967132   <- new


  xgb2025    0.967099   <- new


  xgb13      0.967111   <- new


In [3]:
def run(cols):
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    cf = np.zeros((5, len(cols)))
    for f in range(5):
        tr, va = folds != f, folds == f
        clf = LogisticRegression(C=1.0, max_iter=2000).fit(Loof[np.ix_(tr, cols)],
                                                           y[tr])
        oof[va] = clf.decision_function(Loof[np.ix_(va, cols)])
        tst[f] = clf.decision_function(Ltest[:, cols])
        cf[f] = clf.coef_[0]
    per = np.array([roc_auc_score(y[folds == f], oof[folds == f]) for f in range(5)])
    return per, tst, cf


per25, test25, coef25 = run(KEEP25)
per29, test29, coef29 = run(list(range(len(names))))

ROW39_CV = 0.967873
repro = per25.mean() - ROW39_CV
REPRODUCED = abs(repro) < 1e-4

print(f"{'':22} {'fold 0':>9} {'fold 1':>9} {'fold 2':>9} {'fold 3':>9} {'fold 4':>9}")
for lbl, p in (("25 members, row 39", per25), ("29 members, this run", per29)):
    print(f"{lbl:22} " + " ".join(f"{v:9.6f}" for v in p))
print()
print(f"25-member CV {per25.mean():.6f} +/- {per25.std():.6f}"
      f"   (row 39 recorded {ROW39_CV:.6f}, diff {repro:+.2e})")
print(f"29-member CV {per29.mean():.6f} +/- {per29.std():.6f}")
if not REPRODUCED:
    print("\nROW 39 DID NOT REPRODUCE. Nothing below is comparable to it.")

                          fold 0    fold 1    fold 2    fold 3    fold 4
25 members, row 39      0.967251  0.967991  0.968175  0.968413  0.967534
29 members, this run    0.967287  0.968045  0.968219  0.968474  0.967601

25-member CV 0.967873 +/- 0.000424   (row 39 recorded 0.967873, diff -2.07e-07)
29-member CV 0.967925 +/- 0.000428


In [4]:
def paired(a, b, lbl):
    d = a - b
    t = d.mean() / (d.std(ddof=1) / np.sqrt(len(d)))
    print(f"{lbl:38} {d.mean():+.6f}  sd {d.std(ddof=1):.6f}  "
          f"{(d > 0).sum()}/5  t(4)={t:.2f}")
    print("     per fold: " + "  ".join(f"{v:+.6f}" for v in d))
    return d


d_new = paired(per29, per25, "29 vs 25 members (row 39)")
print()
print("For scale, the same experiment for CatBoost:")
print("  19 -> 23, four more CatBoost seeds  +0.000014  (row 32)")
print("The prediction in the header was +0.000014 to +0.000050, by splitting.")

GATE_FLOOR = 5e-05
wins, mean_g = int((d_new > 0).sum()), float(d_new.mean())
print()
if not REPRODUCED:
    print("VERDICT: blocked, row 39 did not reproduce")
elif wins >= 4 and mean_g >= GATE_FLOOR:
    print("VERDICT: the four seeds earn their place on the pre-registered gate.")
elif wins >= 4 and mean_g > 0:
    print("VERDICT: real and negligible, the same call row 32 got. Under the floor, so")
    print("  it is logged as certain-and-small rather than as an improvement.")
else:
    print("VERDICT: the four seeds add nothing. XGBoost seed averaging is closed.")

29 vs 25 members (row 39)              +0.000052  sd 0.000013  5/5  t(4)=9.16
     per fold: +0.000036  +0.000054  +0.000044  +0.000062  +0.000067

For scale, the same experiment for CatBoost:
  19 -> 23, four more CatBoost seeds  +0.000014  (row 32)
The prediction in the header was +0.000014 to +0.000050, by splitting.

VERDICT: the four seeds earn their place on the pre-registered gate.


In [5]:
# Did the combiner split the existing weight or add to it? Row 32's finding was that
# five near-identical members share one member's weight rather than multiplying it.
was = {names[i]: coef25[:, i].mean() for i in range(len(KEEP25))}
XG = ["xgb_te", "xgb2024", "xgb7", "xgb2025", "xgb13"]

print(f"{'member':10} {'29-member':>11} {'fold sd':>9}   {'row 39':>9}")
for i in np.argsort(-coef29.mean(axis=0)):
    n = names[i]
    tail = f"{'new':>9}" if n in NEW else f"{was[n]:>+9.4f}"
    print(f"{n:10} {coef29[:, i].mean():>+11.4f} {coef29[:, i].std():>9.4f}   {tail}")

xg_sum = sum(coef29[:, names.index(n)].mean() for n in XG)
print(f"\nsum of the 5 XGBoost coefficients : {xg_sum:+.4f}")
print(f"xgb_te alone in row 39            : {was['xgb_te']:+.4f}")
print(f"difference                        : {xg_sum - was['xgb_te']:+.4f}")
print("Row 32's equivalent: five CatBoost coefficients summed to +0.4016 against the")
print("single member's +0.3512, so the five were worth +0.0504 more than the one.")
print(f"largest fold-to-fold sd: {coef29.std(axis=0).max():.4f}")

member       29-member   fold sd      row 39
neural_te      +0.1339    0.0062     +0.1316
xgb2024        +0.1173    0.0062         new
lr003          +0.1025    0.0267     +0.1084
xgb7           +0.1005    0.0120         new
xgb13          +0.0951    0.0039         new
lr005          +0.0842    0.0132     +0.0906
neural         +0.0813    0.0025     +0.0842
xgb2025        +0.0795    0.0074         new
xgb_te         +0.0749    0.0122     +0.2751
bag7           +0.0746    0.0135     +0.0766
bag13          +0.0715    0.0050     +0.0741
bag42          +0.0706    0.0120     +0.0784
cat2024        +0.0620    0.0088     +0.0699
bag2025        +0.0496    0.0060     +0.0519
cat13          +0.0493    0.0073     +0.0573
cat2025        +0.0482    0.0033     +0.0559
cat42          +0.0437    0.0051     +0.0503
cat7           +0.0368    0.0078     +0.0439
bag2024        +0.0171    0.0326     +0.0158
te13           +0.0081    0.0075     +0.0389
trees1000      +0.0051    0.0085     +0.0051
te2024    

In [6]:
pred = test29.mean(axis=0)
prob = 1 / (1 + np.exp(-pred))

prev = pd.read_csv(S / "stack_oof_25.csv")
assert (prev["id"].to_numpy() == test["id"].to_numpy()).all()
t39 = pd.Series(pred).corr(pd.Series(prev["addicted_label"].to_numpy()),
                           method="spearman")
print(f"spearman vs row 39 on disk : {t39:.7f}")

out = S / "stack_oof_29.csv"
sub = pd.DataFrame({"id": test["id"], "addicted_label": prob})
assert len(sub) == len(test) and sub["addicted_label"].between(0, 1).all()
sub.to_csv(out, index=False)
print(f"\nwrote {out.name}, {len(sub):,} rows")
print(f"ledger: CV {per29.mean():.6f} +/- {per29.std():.6f}, "
      f"vs row 39 {mean_g:+.6f} ({wins}/5, sd {d_new.std(ddof=1):.6f})")

spearman vs row 39 on disk : 0.9998940



wrote stack_oof_29.csv, 296,302 rows
ledger: CV 0.967925 +/- 0.000428, vs row 39 +0.000052 (5/5, sd 0.000013)
